In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

np.random.seed(42)

# File paths

SPLIT_DIR = Path("../backend/data/splits_70_15_15")
TRAIN_PATH = SPLIT_DIR / "train.parquet"
VAL_PATH = SPLIT_DIR / "val.parquet"
TEST_PATH = SPLIT_DIR / "test.parquet"

OUT_DIR = Path("../backend/data/KF_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output tag 
MODEL_TAG = "kalman filter"


# Configuration

OBS_COLS = ["temperature", "humidity", "audio_density"]
D = len(OBS_COLS)

BASE_DT_MIN = 15.0
USE_DT_AWARE_Q = True

MIN_ROWS_VAL = 200
MIN_ROWS_TEST = 200

Q_SCALES = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0]
S_JITTER = 1e-6

A = np.eye(D)
H = np.eye(D)

# Helpers

def sym(M: np.ndarray) -> np.ndarray:
    """Symmetrize a square matrix to reduce numerical asymmetry."""
    return 0.5 * (M + M.T)

def safe_solve(A_mat, b_vec):
    """
    Solve A x = b robustly.
    Falls back to pseudo-inverse if A is singular/ill-conditioned.
    """
    try:
        return np.linalg.solve(A_mat, b_vec)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A_mat) @ b_vec

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize types, drop invalid rows, and sort by hive/time.
    Ensures: published_at is UTC datetime, tag_number is integer-like, obs cols are numeric.
    """
    df = df.copy()
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True, errors="coerce")
    df["tag_number"] = pd.to_numeric(df["tag_number"], errors="coerce").astype("Int64")
    for c in OBS_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["published_at", "tag_number"])
    return df.sort_values(["tag_number", "published_at"]).reset_index(drop=True)

def usable_rows(df_h: pd.DataFrame) -> int:
    """Count rows that have at least one finite observation among OBS_COLS."""
    z = df_h[OBS_COLS].to_numpy(float)
    return int(np.isfinite(z).any(axis=1).sum())

def init_x0(z: np.ndarray) -> np.ndarray:
    """Initialize the state with the first available (finite) observation per dimension."""
    x0 = np.zeros(D, float)
    for t in range(len(z)):
        m = np.isfinite(z[t])
        if m.any():
            x0[m] = z[t, m]
            break
    return x0

def persistence_forecast_missing_safe(z: np.ndarray) -> np.ndarray:
    """
    Baseline forecast: persistence using the most recently observed value per dimension.
    Produces NaN until the first observation arrives for that dimension.
    """
    T, d = z.shape
    out = np.full((T, d), np.nan, float)
    last = np.full((d,), np.nan, float)
    for t in range(T):
        out[t] = last
        m = np.isfinite(z[t])
        last[m] = z[t, m]
    return out

def rmse_per_dim(y: np.ndarray, yhat: np.ndarray) -> np.ndarray:
    """Compute RMSE per dimension, ignoring NaNs."""
    y = np.asarray(y, float)
    yhat = np.asarray(yhat, float)
    out = np.full((y.shape[1],), np.nan, float)
    for d in range(y.shape[1]):
        m = np.isfinite(y[:, d]) & np.isfinite(yhat[:, d])
        if m.sum() > 0:
            out[d] = float(np.sqrt(np.mean((y[m, d] - yhat[m, d]) ** 2)))
    return out

def percentile_threshold(arr, p=95.0, min_n=500):
    """Return the p-th percentile threshold if there are at least min_n finite samples."""
    a = np.asarray(arr, float)
    a = a[np.isfinite(a)]
    if a.size < min_n:
        return np.nan
    return float(np.percentile(a, p))

# Chi-square quantiles for NIS calibration (DOF-dependent)

def _chi2_ppf_wilson_hilferty(p: float, k: int) -> float:
    """
    Approximate chi-square inverse CDF using the Wilson–Hilferty transform.
    Suitable for small DOF (here k <= D = 3).
    """
    # Prefer scipy's inverse normal if available; otherwise use a standard approximation.
    try:
        from scipy.stats import norm
        z = float(norm.ppf(p))
    except Exception:
        # Peter J. Acklam's inverse normal CDF approximation (practical accuracy).
        def inv_norm_cdf(pp):
            a = [-3.969683028665376e+01, 2.209460984245205e+02,
                 -2.759285104469687e+02, 1.383577518672690e+02,
                 -3.066479806614716e+01, 2.506628277459239e+00]
            b = [-5.447609879822406e+01, 1.615858368580409e+02,
                 -1.556989798598866e+02, 6.680131188771972e+01,
                 -1.328068155288572e+01]
            c = [-7.784894002430293e-03, -3.223964580411365e-01,
                 -2.400758277161838e+00, -2.549732539343734e+00,
                 4.374664141464968e+00, 2.938163982698783e+00]
            d = [7.784695709041462e-03, 3.224671290700398e-01,
                 2.445134137142996e+00, 3.754408661907416e+00]
            plow = 0.02425
            phigh = 1 - plow
            if pp < plow:
                q = np.sqrt(-2 * np.log(pp))
                return (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                       ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            if pp > phigh:
                q = np.sqrt(-2 * np.log(1 - pp))
                return -(((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5]) / \
                        ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1)
            q = pp - 0.5
            r = q * q
            return (((((a[0] * r + a[1]) * r + a[2]) * r + a[3]) * r + a[4]) * r + a[5]) * q / \
                   (((((b[0] * r + b[1]) * r + b[2]) * r + b[3]) * r + b[4]) * r + 1)

        z = float(inv_norm_cdf(p))

    k = max(int(k), 1)
    return float(k * (1 - 2 / (9 * k) + z * np.sqrt(2 / (9 * k))) ** 3)

def chi2_ppf(p: float, k: int) -> float:
    """Prefer scipy.stats.chi2.ppf; otherwise use Wilson–Hilferty approximation."""
    try:
        from scipy.stats import chi2
        return float(chi2.ppf(p, df=int(k)))
    except Exception:
        return _chi2_ppf_wilson_hilferty(p, k)

def build_chi2_thresholds(ps=(0.95, 0.99), max_dof=10):
    """Precompute chi-square thresholds for a set of probabilities and DOF values."""
    out = {}
    for p in ps:
        out[p] = {k: chi2_ppf(p, k) for k in range(1, max_dof + 1)}
    return out


# Gap-aware dt scaling for process noise

def dt_since_last_observed_minutes(df_h: pd.DataFrame, obs_cols: list[str]) -> np.ndarray:
    """
    Compute minutes since the last row that contained ANY observed value.
    During missing stretches, dt increases so Q can scale appropriately.
    """
    has_any = df_h[obs_cols].notna().any(axis=1).to_numpy()
    t = pd.to_datetime(df_h["published_at"], utc=True).to_numpy()

    out = np.full(len(df_h), np.nan, dtype=float)
    last_t = None
    for i in range(len(df_h)):
        if last_t is None:
            out[i] = BASE_DT_MIN
        else:
            out[i] = (t[i] - last_t) / np.timedelta64(1, "m")
        if has_any[i]:
            last_t = t[i]

    out = np.where(np.isfinite(out) & (out > 0), out, BASE_DT_MIN)
    return out

def build_dt_scale(df_h: pd.DataFrame):
    """Return per-timestep scaling for Q based on time gaps between observations."""
    if not USE_DT_AWARE_Q:
        return None
    dt = dt_since_last_observed_minutes(df_h, OBS_COLS)
    s = dt / BASE_DT_MIN
    s = np.where(np.isfinite(s) & (s > 0), s, 1.0)
    return np.clip(s, 1.0, 16.0)

# R estimation (TRAIN-only; robust)

def robust_var_mad(x: np.ndarray) -> float:
    """
    Robust variance estimate using MAD (median absolute deviation).
    Returns NaN if not enough samples are available.
    """
    x = x[np.isfinite(x)]
    if x.size < 100:
        return np.nan
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    sigma = 1.4826 * mad
    return float(max(sigma ** 2, 1e-6))

def estimate_R_from_train(train_df: pd.DataFrame) -> np.ndarray:
    """
    Estimate measurement noise covariance R using TRAIN only.

    Approach:
      - For each hive, compute first differences as a proxy for high-frequency noise
        (plus some dynamics).
      - Use a robust MAD-based variance per dimension.
      - Convert diff variance to measurement variance approximately via /2.
      - Aggregate robustly across hives using the median.
    """
    per_hive = []
    for _, g in train_df.groupby("tag_number"):
        z = g[OBS_COLS].to_numpy(float)
        if len(z) < 200:
            continue
        dz = np.diff(z, axis=0)
        v = []
        for d in range(D):
            vd = robust_var_mad(dz[:, d])
            if np.isfinite(vd):
                vd = vd / 2.0
            v.append(vd)
        per_hive.append(v)

    if len(per_hive) == 0:
        return np.diag([1e-3] * D)

    R_diag = np.nanmedian(np.asarray(per_hive, float), axis=0)
    R_diag = np.where(np.isfinite(R_diag) & (R_diag > 1e-8), R_diag, 1e-3)
    R_diag = np.clip(R_diag, 1e-6, None)
    return np.diag(R_diag)

# Kalman filter (fair pre-update forecast)

def kf_run_fair(z: np.ndarray, Q_base: np.ndarray, R: np.ndarray,
               x0: np.ndarray, P0: np.ndarray, dt_scale=None):
    """
    Run a Kalman filter that evaluates the one-step-ahead forecast fairly:
      - Predict step produces x_pred[t] BEFORE assimilating z[t].
      - Update step assimilates only observed dimensions of z[t].
    """
    T = len(z)
    x = x0.copy()
    P = P0.copy()

    x_pred = np.zeros((T, D), float)
    P_pred = np.zeros((T, D, D), float)
    x_filt = np.zeros((T, D), float)

    nis = np.full(T, np.nan, float)
    nis_norm = np.full(T, np.nan, float)
    nis_dof = np.zeros(T, int)

    I = np.eye(D)

    for t in range(T):
        scale = 1.0 if dt_scale is None else float(dt_scale[t])
        Q = Q_base * scale

        # Predict (forecast BEFORE using z[t])
        x = A @ x
        P = A @ P @ A.T + Q
        P = sym(P)

        x_pred[t] = x
        P_pred[t] = P

        # Update (assimilate observed dimensions only)
        zt = z[t]
        m = np.isfinite(zt)
        m_dim = int(m.sum())
        nis_dof[t] = m_dim

        if m_dim > 0:
            Ht = H[m]
            Rt = R[np.ix_(m, m)]
            y = zt[m] - (Ht @ x)

            S = Ht @ P @ Ht.T + Rt
            S = sym(S) + S_JITTER * np.eye(m_dim)

            Sinv_y = safe_solve(S, y)
            nis_val = float(y.T @ Sinv_y)
            nis[t] = nis_val
            nis_norm[t] = nis_val / max(m_dim, 1)

            K = (P @ Ht.T) @ safe_solve(S, np.eye(m_dim))

            x = x + K @ y
            P = (I - K @ Ht) @ P @ (I - K @ Ht).T + K @ Rt @ K.T
            P = sym(P)

        x_filt[t] = x

    return x_pred, x_filt, P_pred, nis, nis_norm, nis_dof

# Main pipeline

def main():
    print("Loading data splits...")
    train_df = clean_df(pd.read_parquet(TRAIN_PATH))
    val_df = clean_df(pd.read_parquet(VAL_PATH))
    test_df = clean_df(pd.read_parquet(TEST_PATH))

    print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
    print(
        "Hives (train/val/test):",
        train_df["tag_number"].nunique(),
        val_df["tag_number"].nunique(),
        test_df["tag_number"].nunique(),
    )

    # Training standard deviation for normalized aggregation of RMSE across dimensions
    train_std = train_df[OBS_COLS].astype(float).std(skipna=True).to_numpy()
    train_std = np.where(np.isfinite(train_std) & (train_std > 1e-12), train_std, 1.0)

    def agg_zrmse(rmse_raw: np.ndarray) -> float:
        """Aggregate RMSE across dimensions using training standard deviation normalization."""
        rmse_raw = np.asarray(rmse_raw, float)
        m = np.isfinite(rmse_raw) & np.isfinite(train_std) & (train_std > 1e-12)
        if m.sum() == 0:
            return np.nan
        return float(np.mean(rmse_raw[m] / train_std[m]))

    # Estimate measurement noise R using TRAIN only
    R = estimate_R_from_train(train_df)
    R_diag = np.diag(R)
    print("\nR diagonal (TRAIN-only estimate):", R_diag)

   
    # Select evaluation/tuning hives from VAL only (no TEST involvement)
  
    usable_val_hives = sorted([
        int(h) for h in val_df["tag_number"].dropna().unique()
        if usable_rows(val_df[val_df["tag_number"] == h]) >= MIN_ROWS_VAL
    ])

    print("\nUsable VAL hives:", len(usable_val_hives))

    if len(usable_val_hives) == 0:
        raise ValueError("No usable VAL hives. Lower MIN_ROWS_VAL or verify your data splits.")

    # Persist the validation-selected hive set used for tuning
    pd.DataFrame({"hive_id": usable_val_hives}).to_csv(
        OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv", index=False
    )

    # Reporting-only: summarize TEST availability without using it for selection/tuning
    usable_test_hives = sorted([
        int(h) for h in test_df["tag_number"].dropna().unique()
        if usable_rows(test_df[test_df["tag_number"] == h]) >= MIN_ROWS_TEST
    ])
    common_val_test = sorted(list(set(usable_val_hives) & set(test_df["tag_number"].dropna().astype(int).unique())))
    common_val_test_usable = sorted(list(set(usable_val_hives) & set(usable_test_hives)))

    print("Usable TEST hives (reporting only):", len(usable_test_hives))
    print("VAL-selected hives present in TEST:", len(common_val_test))
    print("VAL-selected hives that are TEST-usable:", len(common_val_test_usable))

   
    # Tune Q on VAL (fair forecast), using the VAL-selected hive set
  
    print("\nTuning Q on VAL (FAIR pre-update forecast), using VAL-only hive set...")
    tuning_rows = []
    P0 = np.eye(D) * 10.0

    for s in Q_SCALES:
        Q = np.diag(np.clip(R_diag * float(s), 1e-8, None))
        scores = []
        for h in usable_val_hives:
            vh = val_df[val_df["tag_number"] == h]
            z = vh[OBS_COLS].to_numpy(float)

            x_pred, *_ = kf_run_fair(
                z, Q, R, init_x0(z), P0,
                dt_scale=build_dt_scale(vh),
            )
            scores.append(agg_zrmse(rmse_per_dim(z, x_pred)))

        tuning_rows.append({
            "Q_scale": float(s),
            "median_val_agg_zrmse": float(np.nanmedian(scores)),
            "mean_val_agg_zrmse": float(np.nanmean(scores)),
            "num_hives": int(len(scores)),
        })

    tuning_df = pd.DataFrame(tuning_rows).sort_values("median_val_agg_zrmse").reset_index(drop=True)
    best_scale = float(tuning_df.iloc[0]["Q_scale"])
    Q_best = np.diag(np.clip(R_diag * best_scale, 1e-8, None))

    tuning_df.to_csv(OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv", index=False)
    print("\nBest Q_scale:", best_scale)
    print(tuning_df)


    # Run final KF on VAL + TEST (store per-step outputs)
    
    print("\nRunning final KF on VAL + TEST (VAL-selected hives)...")
    per_step_frames = []

    for split_name, df_split in [("val", val_df), ("test", test_df)]:
        # Run only on hives selected from VAL, and only if the hive exists in the current split
        present_hives = set(df_split["tag_number"].dropna().astype(int).unique())
        run_hives = [h for h in usable_val_hives if h in present_hives]

        for h in run_hives:
            dh = df_split[df_split["tag_number"] == h]
            if dh.empty:
                continue

            z = dh[OBS_COLS].to_numpy(float)
            x_pred, x_filt, P_pred, nis, nis_norm, nis_dof = kf_run_fair(
                z, Q_best, R, init_x0(z), P0,
                dt_scale=build_dt_scale(dh),
            )

            pred_std = np.sqrt(np.maximum(np.diagonal(P_pred, axis1=1, axis2=2), 0.0))

            per_step_frames.append(pd.DataFrame({
                "published_at": pd.to_datetime(dh["published_at"].to_numpy(), utc=True),
                "hive_id": int(h),
                "split": split_name,

                "y_true_temperature": z[:, 0],
                "y_true_humidity": z[:, 1],
                "y_true_audio_density": z[:, 2],

                "y_pred_temperature": x_pred[:, 0],
                "y_pred_humidity": x_pred[:, 1],
                "y_pred_audio_density": x_pred[:, 2],

                "pred_std_temperature": pred_std[:, 0],
                "pred_std_humidity": pred_std[:, 1],
                "pred_std_audio_density": pred_std[:, 2],

                "kf_filt_temperature": x_filt[:, 0],
                "kf_filt_humidity": x_filt[:, 1],
                "kf_filt_audio_density": x_filt[:, 2],

                "nis_raw": nis,
                "nis_norm": nis_norm,
                "nis_dof": nis_dof,

                # Reporting flags
                "is_val_selected_hive": True,
                "is_test_usable_hive": (int(h) in set(usable_test_hives)),
            }))

    out_df = pd.concat(per_step_frames, ignore_index=True)

   
    # Anomaly thresholds (percentile-based) and calibration checks (chi-square-based)
    
    # Percentile thresholds learned on VAL only (normalized NIS, dof>0 only)
    val_mask = (out_df["split"] == "val") & (out_df["nis_dof"] > 0) & np.isfinite(out_df["nis_norm"])
    val_scores = out_df.loc[val_mask, "nis_norm"].to_numpy(float)

    thr95_pct = percentile_threshold(val_scores, 95.0, min_n=500)
    thr99_pct = percentile_threshold(val_scores, 99.0, min_n=500)

    print("\nVAL-only percentile thresholds for nis_norm (dof>0 only):")
    print(" thr95_pct:", thr95_pct, " thr99_pct:", thr99_pct, " | samples:", int(np.isfinite(val_scores).sum()))

    out_df["is_anomaly_norm_pct_p95"] = (
        (out_df["nis_dof"] > 0)
        & np.isfinite(out_df["nis_norm"])
        & np.isfinite(thr95_pct)
        & (out_df["nis_norm"] > thr95_pct)
    ) if np.isfinite(thr95_pct) else np.zeros(len(out_df), dtype=bool)

    out_df["is_anomaly_norm_pct_p99"] = (
        (out_df["nis_dof"] > 0)
        & np.isfinite(out_df["nis_norm"])
        & np.isfinite(thr99_pct)
        & (out_df["nis_norm"] > thr99_pct)
    ) if np.isfinite(thr99_pct) else np.zeros(len(out_df), dtype=bool)

    # Chi-square thresholds for RAW NIS (classical consistency check): nis_raw ~ chi2(dof)
    chi2_thr = build_chi2_thresholds(ps=(0.95, 0.99), max_dof=max(D, 3))

    def map_chi2(p, dof):
        dof = int(dof)
        if dof <= 0:
            return np.nan
        return chi2_thr[p].get(dof, chi2_ppf(p, dof))

    out_df["chi2_thr_raw_p95"] = out_df["nis_dof"].apply(lambda k: map_chi2(0.95, k))
    out_df["chi2_thr_raw_p99"] = out_df["nis_dof"].apply(lambda k: map_chi2(0.99, k))

    out_df["is_inconsistent_chi2_p95"] = (
        (out_df["nis_dof"] > 0)
        & np.isfinite(out_df["nis_raw"])
        & np.isfinite(out_df["chi2_thr_raw_p95"])
        & (out_df["nis_raw"] > out_df["chi2_thr_raw_p95"])
    )

    out_df["is_inconsistent_chi2_p99"] = (
        (out_df["nis_dof"] > 0)
        & np.isfinite(out_df["nis_raw"])
        & np.isfinite(out_df["chi2_thr_raw_p99"])
        & (out_df["nis_raw"] > out_df["chi2_thr_raw_p99"])
    )

    
    # Save per-step outputs
   
    out_df.to_parquet(OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet", index=False)
    out_df[out_df["split"] == "val"].to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_val.parquet", index=False)
    out_df[out_df["split"] == "test"].to_parquet(OUT_DIR / f"{MODEL_TAG}_forecasts_test.parquet", index=False)

 
    # Summary metrics per hive + calibration summaries
    
    summary_rows = []
    calib_rows = []

    for split_name in ["val", "test"]:
        df_split = out_df[out_df["split"] == split_name].copy()

        for hive_id, g in df_split.groupby("hive_id"):
            z = g[["y_true_temperature", "y_true_humidity", "y_true_audio_density"]].to_numpy(float)
            pred = g[["y_pred_temperature", "y_pred_humidity", "y_pred_audio_density"]].to_numpy(float)
            base = persistence_forecast_missing_safe(z)

            rmse_base = rmse_per_dim(z, base)
            rmse_kf = rmse_per_dim(z, pred)

            summary_rows.append({
                "hive_id": int(hive_id),
                "split": split_name,
                "baseline_agg_zrmse": agg_zrmse(rmse_base),
                "kf_agg_zrmse": agg_zrmse(rmse_kf),
                "baseline_rmse_temp": rmse_base[0],
                "baseline_rmse_humid": rmse_base[1],
                "baseline_rmse_audio": rmse_base[2],
                "kf_rmse_temp": rmse_kf[0],
                "kf_rmse_humid": rmse_kf[1],
                "kf_rmse_audio": rmse_kf[2],
                "n_rows": int(len(g)),
                "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]) if "is_test_usable_hive" in g else False,
            })

            # Consistency summaries (chi-square based): exceedance rates for dof>0 rows
            m = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
            if m.any():
                p95_rate = float(np.mean(g.loc[m, "nis_raw"].to_numpy() > g.loc[m, "chi2_thr_raw_p95"].to_numpy()))
            else:
                p95_rate = np.nan

            m = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
            if m.any():
                p99_rate = float(np.mean(g.loc[m, "nis_raw"].to_numpy() > g.loc[m, "chi2_thr_raw_p99"].to_numpy()))
            else:
                p99_rate = np.nan

            calib_rows.append({
                "hive_id": int(hive_id),
                "split": split_name,
                "chi2_exceed_rate_p95": p95_rate,  # expected ~0.05 when well-calibrated
                "chi2_exceed_rate_p99": p99_rate,  # expected ~0.01 when well-calibrated
                "n_dof_pos": int((g["nis_dof"] > 0).sum()),
                "is_test_usable_hive": bool(g["is_test_usable_hive"].iloc[0]) if "is_test_usable_hive" in g else False,
            })

    summary_df = pd.DataFrame(summary_rows)
    calib_df = pd.DataFrame(calib_rows)

    summary_df.to_csv(OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv", index=False)
    calib_df.to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv", index=False)

    # Overall calibration summary per split
    overall_calib = []
    for split_name in ["val", "test"]:
        g = out_df[out_df["split"] == split_name]
        m95 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p95"])
        m99 = (g["nis_dof"] > 0) & np.isfinite(g["nis_raw"]) & np.isfinite(g["chi2_thr_raw_p99"])
        overall_calib.append({
            "split": split_name,
            "chi2_exceed_rate_p95_overall": float(np.mean(g.loc[m95, "nis_raw"] > g.loc[m95, "chi2_thr_raw_p95"])) if m95.any() else np.nan,
            "chi2_exceed_rate_p99_overall": float(np.mean(g.loc[m99, "nis_raw"] > g.loc[m99, "chi2_thr_raw_p99"])) if m99.any() else np.nan,
            "n_samples_dof_pos": int((g["nis_dof"] > 0).sum()),
        })
    pd.DataFrame(overall_calib).to_csv(OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv", index=False)

    # Record run configuration for reproducibility
    pd.DataFrame([{
        "MODEL_TAG": MODEL_TAG,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "EVAL_MODE": "FAIR_pre_update_forecast",
        "R_SOURCE": "TRAIN_only (robust diff-MAD proxy)",
        "Q_TUNED_ON": "VAL_only (VAL-selected hives only; no test-informed selection)",
        "VAL_HIVE_SELECTION": f"usable_rows>=MIN_ROWS_VAL ({MIN_ROWS_VAL})",
        "TEST_USABLE_CRITERIA_REPORTING_ONLY": f"usable_rows>=MIN_ROWS_TEST ({MIN_ROWS_TEST})",
        "best_Q_scale": best_scale,
        "USE_DT_AWARE_Q": USE_DT_AWARE_Q,
        "R_diag": ",".join([f"{x:.6g}" for x in R_diag]),

        # Percentile anomaly thresholds (data-driven)
        "nis_norm_pct_p95_val": thr95_pct,
        "nis_norm_pct_p99_val": thr99_pct,

        # Chi-square thresholds (theoretical consistency)
        "chi2_raw_p95_dof1": chi2_thr[0.95].get(1, np.nan),
        "chi2_raw_p95_dof2": chi2_thr[0.95].get(2, np.nan),
        "chi2_raw_p95_dof3": chi2_thr[0.95].get(3, np.nan),
        "chi2_raw_p99_dof1": chi2_thr[0.99].get(1, np.nan),
        "chi2_raw_p99_dof2": chi2_thr[0.99].get(2, np.nan),
        "chi2_raw_p99_dof3": chi2_thr[0.99].get(3, np.nan),

        "eval_hives_val_count": len(usable_val_hives),
        "val_selected_hives_present_in_test": len(common_val_test),
        "val_selected_hives_test_usable_count": len(common_val_test_usable),

        "seed": 42,
    }]).to_csv(OUT_DIR / f"{MODEL_TAG}_run_config.csv", index=False)

    print("\nSaved outputs to:", OUT_DIR)
    print(" -", OUT_DIR / f"{MODEL_TAG}_eval_hives_val_only.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_q_tuning_results.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_val_test_forecast_nis.parquet")
    print(" -", OUT_DIR / f"{MODEL_TAG}_summary_metrics_per_hive.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_per_hive.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_nis_chi2_calibration_overall.csv")
    print(" -", OUT_DIR / f"{MODEL_TAG}_run_config.csv")

if __name__ == "__main__":
    main()

Loading data splits...
Train: (560733, 31) Val: (120160, 31) Test: (120187, 31)
Hives (train/val/test): 52 52 52

R diagonal (TRAIN-only estimate): [0.00175848 0.00395658 1.08537162]

Usable VAL hives: 33
Usable TEST hives (reporting only): 27
VAL-selected hives present in TEST: 33
VAL-selected hives that are TEST-usable: 26

Tuning Q on VAL (FAIR pre-update forecast), using VAL-only hive set...

Best Q_scale: 3.0
   Q_scale  median_val_agg_zrmse  mean_val_agg_zrmse  num_hives
0   3.0000              0.168733            0.176838         33
1   1.0000              0.171089            0.178108         33
2   0.3000              0.184274            0.189832         33
3   0.1000              0.212713            0.215389         33
4   0.0300              0.266478            0.264419         33
5   0.0100              0.336922            0.327761         33
6   0.0030              0.412254            0.404700         33
7   0.0010              0.480247            0.467862         33
8   0.